# 00: Data acquisition

Fetches the canonical PaySim dataset (`ealaxi/paysim1`) and refuses to proceed unless it is byte-for-byte the file the literature used.

**Why the validation is strict.** Every threshold in this study is derived from the class prior. `R = N_neg/N_pos = 773.70` sets the severity ratio, and the Bayes threshold `λ = 1/(1+R) = 0.00129` *is* the prior. A truncated mirror or a resampled copy would silently change both, and every downstream number would be wrong in a way that produces no error message. So the load path asserts the exact row count and fraud count and fails loudly otherwise.

This is also the file Lokanan (2023) cites in his Data Availability Statement and the one Thar & Wai (2025) use, which is what makes the replication in notebook 05 and the benchmark check in notebook 03 legitimate comparisons.

In [1]:
import sys, subprocess, zipfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from momo_fraud import constants as C
from momo_fraud import data as D

RAW_PATH = D.RAW_DIR / C.RAW_FILENAME
D.RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"project root : {PROJECT_ROOT}")
print(f"expecting    : {RAW_PATH}")
print(f"present      : {RAW_PATH.exists()}")

project root : C:\Users\siphe\source\repos\CodeHermez\momo-fraud-supervised-ml
expecting    : C:\Users\siphe\source\repos\CodeHermez\momo-fraud-supervised-ml\data\raw\PS_20174392719_1491204439457_log.csv
present      : True


## Download

Tries the Kaggle API first; if credentials are missing it prints instructions rather than failing.

Kaggle supports two credential formats and the CLI checks them in this order:

1. **`~/.kaggle/access_token`**: a bare `KGAT_…` token on one line. This is what *Settings → API → Create New Token* now hands you.
2. **`~/.kaggle/kaggle.json`**: the older `{"username": …, "key": …}` file.

Either works. On Windows `~` resolves to `C:\Users\<you>`, and note that Kaggle also accepts `access_token.txt`: a concession to Notepad, which appends `.txt` whether you want it or not.

In [2]:
KAGGLE_DATASET = "ealaxi/paysim1"

# Checked in the same order the CLI checks them.
CREDENTIAL_PATHS = [
    Path.home() / ".kaggle" / "access_token",
    Path.home() / ".kaggle" / "access_token.txt",
    Path.home() / ".kaggle" / "kaggle.json",
]
credentials = next((p for p in CREDENTIAL_PATHS if p.exists()), None)

if RAW_PATH.exists():
    print(f"Already present ({RAW_PATH.stat().st_size / 1e6:.0f} MB): skipping download.")

elif credentials is not None:
    print(f"Using credentials at {credentials}")
    print(f"Downloading {KAGGLE_DATASET} (~186 MB) …")
    subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", KAGGLE_DATASET, "-p", str(D.RAW_DIR), "--unzip"],
        check=True,
    )
    # --unzip usually cleans up after itself; catch the case where it does not.
    for archive in D.RAW_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(D.RAW_DIR)
        archive.unlink()
    print("Done.")

else:
    print(
        "No Kaggle credentials found. Looked for:\n"
        + "".join(f"  {p}\n" for p in CREDENTIAL_PATHS)
        + "\nEither:\n"
        "  (a) generate a token at kaggle.com/settings/api and save it as a single\n"
        "      line in ~/.kaggle/access_token, then re-run this cell; or\n"
        f"  (b) download {KAGGLE_DATASET} manually and unzip it so that\n"
        f"      {RAW_PATH}\n"
        "      exists, then continue from the next cell."
    )

Already present (494 MB)  skipping download.


## Load and validate

`load_raw` reads straight into compact dtypes (float32 balances, int16 step, categorical type)  roughly half the memory of the default inference  then asserts the canonical shape.

In [3]:
df = D.load_raw(RAW_PATH)   # raises DatasetValidationError on any mismatch
print(f"Validated: {len(df):,} rows x {df.shape[1]} columns")
print(f"Memory:    {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

Validated: 6,362,620 rows x 11 columns
Memory:    0.39 GB


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.639648,C1231006815,170136.0,160296.359375,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,C1666544295,21249.0,19384.720703,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.000000,C1305486145,181.0,0.000000,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.000000,C840083671,181.0,0.000000,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.139648,C2048537720,41554.0,29885.859375,M1230701703,0.0,0.0,0,0


In [4]:
summary = D.summarise(df)
for key, value in summary.items():
    formatted = f"{value:,.6f}".rstrip("0").rstrip(".") if isinstance(value, float) else f"{value:,}"
    print(f"  {key:22s} {formatted}")

print(f"\nSeverity ratio R = N_neg/N_pos = {summary['imbalance_ratio']:.2f}")
print(f"Bayes threshold  = 1/(1+R)     = {1 / (1 + summary['imbalance_ratio']):.6f}")
print(f"Class prior                    = {summary['prevalence']:.6f}")
print("\nThe last two agree by construction: Johnson & Khoshgoftaar's result that")
print("the recommended cost ratio places the decision boundary exactly at the prior.")

  n_transactions         6,362,620
  n_fraud                8,213
  n_legitimate           6,354,407
  prevalence             0.001291
  imbalance_ratio        773.701084
  n_steps                743
  n_days                 30.958333

Severity ratio R = N_neg/N_pos = 773.70
Bayes threshold  = 1/(1+R)     = 0.001291
Class prior                    = 0.001291

The last two agree by construction  Johnson & Khoshgoftaar's result that
the recommended cost ratio places the decision boundary exactly at the prior.


## Cache as parquet

Every later notebook loads from this cache. Also records the source file's SHA-256, which goes into the model card in notebook 08 so any exported artifact can be traced to the exact bytes it was trained on.

In [5]:
parquet_path = D.to_parquet(df)
checksum = D.file_hash(RAW_PATH)

print(f"cached      : {parquet_path}")
print(f"parquet size: {parquet_path.stat().st_size / 1e6:.0f} MB")
print(f"csv sha256  : {checksum}")

from momo_fraud import evaluate as E
E.save_json(
    {"dataset": KAGGLE_DATASET, "csv_sha256": checksum, **summary},
    "00_dataset_provenance",
)

cached      : C:\Users\siphe\source\repos\CodeHermez\momo-fraud-supervised-ml\data\processed\paysim.parquet
parquet size: 230 MB
csv sha256  : 16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b


WindowsPath('C:/Users/siphe/source/repos/CodeHermez/momo-fraud-supervised-ml/results/00_dataset_provenance.json')

In [6]:
reloaded = D.load()   # validates again, from the parquet path this time
assert len(reloaded) == C.N_ROWS and int(reloaded[C.TARGET].sum()) == C.N_FRAUD
print("Cache verified. Continue to 01_eda.ipynb.")

Cache verified. Continue to 01_eda.ipynb.
